# 03. Model Storage & Staging-Capacity Gate Downloader

Evaluates authoritative Google Drive API account quota, Colab local NVMe disk capacity, and staging strategy before initiating the 142-shard model transfer into `/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model`.

### Step 1: Pre-Download Staging-Capacity Gate

In [ ]:
import os
import sys
import shutil
import subprocess
import json

# Ensure repository is present in Colab
REPO_DIR = '/content/glm52-drive-runtime'
if not os.path.exists(REPO_DIR):
    print(f"Cloning GLM-5.2 repository into {REPO_DIR}...")
    subprocess.run(['git', 'clone', 'https://github.com/Aqib2607/AI.git', REPO_DIR], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from scripts.drive_check import get_drive_service, get_drive_storage_quota, evaluate_storage_gate

# Model Specifications
MODEL_REPO = 'mastouri/GLM-5.2-colibri-int4-g64-with-int8-mtp'
MODEL_SIZE_GIB = 399.79
MODEL_SIZE_GB = 429.28
REQUIRED_DRIVE_GB = 400.0
RECOMMENDED_DRIVE_GB = 450.0
TEMP_CHUNK_BUFFER_GIB = 3.0

# Paths
DRIVE_MODEL_DIR = '/content/drive/MyDrive/AI - Google Drive/GLM-5.2/model'
LOCAL_DIR = '/content'

# 1. Query Authoritative Google Drive Account Quota via Drive API v3
service = get_drive_service()
quota_info = get_drive_storage_quota(service=service)
drive_free_gb = quota_info.get('free_gb')
drive_limit_gb = quota_info.get('limit_gb')
drive_used_gb = quota_info.get('usage_gb')
is_unlimited = quota_info.get('is_unlimited', False)

# Evaluate Storage Gate
gate_status, gate_reason = evaluate_storage_gate(
    account_free_gb=drive_free_gb,
    is_unlimited=is_unlimited,
    required_gb=REQUIRED_DRIVE_GB,
    recommended_gb=RECOMMENDED_DRIVE_GB
)

# 2. Measure Colab Local Ephemeral NVMe Storage
try:
    l_total, l_used, l_free = shutil.disk_usage(LOCAL_DIR)
    local_free_gib = round(l_free / (1024 ** 3), 2)
    local_total_gib = round(l_total / (1024 ** 3), 2)
except Exception as e:
    local_free_gib = 0.0
    local_total_gib = 0.0

# 3. Decision Logic & Architecture Routing
drive_pass = gate_status in ('GO', 'GO_WITH_LOW_MARGIN', 'GO_WITH_RECOMMENDED_MARGIN', 'GO_UNLIMITED_QUOTA')
can_full_stage_locally = local_free_gib >= (MODEL_SIZE_GIB + 10.0)

if can_full_stage_locally:
    selected_architecture = "Option A: Full Local NVMe Staging (100% weights on fast disk)"
    local_required_gib = MODEL_SIZE_GIB
else:
    selected_architecture = "Option B: Hybrid Staging & Dual-Drive Mirroring (COLI_MODEL_MIRROR)"
    local_required_gib = 15.0  # MTP head (9.3 GB) + hot cache buffer

go_decision = drive_pass and (local_free_gib >= local_required_gib)

# 4. Render Preflight Storage Gate Report
print("=" * 75)
print("         GLM-5.2 COLIBRI STAGING-CAPACITY GATE PREFLIGHT AUDIT")
print("=" * 75)
print(f"Model Repository:                 {MODEL_REPO}")
print(f"Verified Model Size:              {MODEL_SIZE_GIB:.2f} GiB ({MODEL_SIZE_GB:.2f} GB decimal)")
print(f"Temporary Buffer Requirement:     {TEMP_CHUNK_BUFFER_GIB:.2f} GiB")
print("-" * 75)
print("=== Authoritative Google Drive Account Quota (API v3) ===")
print(f"Account Email:                    {quota_info.get('email', 'aqibjawwad2607@gmail.com')}")
print(f"Google Drive Total Plan Quota:    {f'{drive_limit_gb:,.2f} GB' if drive_limit_gb else 'Unlimited'}")
print(f"Google Drive Used Storage:        {f'{drive_used_gb:,.2f} GB' if drive_used_gb else 'Unknown'}")
print(f"Google Drive Available Free:      {f'{drive_free_gb:,.2f} GB' if drive_free_gb else 'Unlimited'}")
print(f"Required Free Storage Threshold:  >= {REQUIRED_DRIVE_GB:.2f} GB (Recommended: >= {RECOMMENDED_DRIVE_GB:.2f} GB)")
print(f"Drive Quota Gate Decision:        {gate_status}")
print(f"Reason:                           {gate_reason}")
print("-" * 75)
print("=== Colab Local Ephemeral NVMe Storage ===")
print(f"Colab Local Available Space:      {local_free_gib:.2f} GiB (Total: {local_total_gib:.2f} GiB)")
print(f"Colab Local Required Space:       {local_required_gib:.2f} GiB (for {selected_architecture.split(':')[0]})")
print(f"Selected Runtime Architecture:    {selected_architecture}")
print("=" * 75)
print(f"PREFLIGHT DECISION:               {'GO - READY FOR DOWNLOAD' if go_decision else 'NO-GO - BLOCKING CAPACITY ISSUE'}")
print("=" * 75)

if not go_decision:
    raise SystemExit(f"Stopping: Storage gate criteria not satisfied ({gate_reason}).")
else:
    print("\n✓ Capacity gate passed. Ready to proceed to Step 2.")

### Step 2: Resumable Model Shard Download

In [ ]:
# Optional: Set your Hugging Face token here if using a gated repository
os.environ['HF_TOKEN'] = 'hf_your_token_here'

# Execute atomic chunk-level downloader via absolute repository path
!python /content/glm52-drive-runtime/scripts/download_model.py \
  --repo "{MODEL_REPO}" \
  --target-dir "{DRIVE_MODEL_DIR}" \
  --min-free-gb 400